# 04 — Inspeção visual do cubo (Etapa 3)

Confere o cubo da sub-AOI antes de extrair embeddings:
1. Séries de patches **com evento** (desmatamento, mineração, fogo) e de um patch **normal**: RGB do S2 e VV do S1 ao longo dos 28 trimestres.
2. **Normalização do TerraMind:** os valores padronizados dos pixels limpos devem ficar numa faixa razoável (média perto de 0, desvio da ordem de 1). Valores muito fora disso indicam unidade errada (offset de +1000, reflectância 0–1, S1 linear em vez de dB).

Roda localmente, lendo o cubo de `data/` (ou no Colab, na mesma sessão que construiu o cubo).

## 1. Setup

In [1]:
import sys, os, subprocess, time
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'github.com/GabrielPow/ProjetoGrassQuality.git'
BRANCH = 'main'
REPO_DIR = '/content/ProjetoGrassQuality'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    url = f'https://{token}@{REPO_URL}' if token else f'https://{REPO_URL}'
    if os.path.exists(REPO_DIR):
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
    else:
        subprocess.run(['git', 'clone', '-b', BRANCH, url, REPO_DIR], check=True)

    from importlib import metadata
    import numpy
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    f'{REPO_DIR}/notebooks/gabriel/requirements-colab.txt'], check=True)
    if metadata.version('numpy') != numpy.__version__:
        print(f'numpy {numpy.__version__} -> {metadata.version("numpy")}: reiniciando o ambiente. '
              'Rode todas as células de novo (a instalação não se repete).')
        os.kill(os.getpid(), 9)
    os.chdir(f'{REPO_DIR}/notebooks/gabriel')
else:
    print('Local: setup do Colab pulado.')

Local: setup do Colab pulado.


In [2]:
import json
import numpy as np
import pandas as pd
import zarr
import config, utils, rotulos, dados

run_dir = utils.new_run('etapa3_inspecao')
root = zarr.open_group(str(config.CUBE_DIR / config.CUBE_PATH_NAME), mode='r')
pids, steps = list(root.attrs['patch_id']), list(root.attrs['steps'])
print(root['s2'].shape, '| tarefas completas:', int(root['done'][:].sum()), 'de', root['done'].size)

(256, 28, 12, 224, 224) | tarefas completas: 896 de 896


## 2. Eventos da sub-AOI (rótulos da Etapa 2)

In [ ]:
lab = config.LABELS_DIR / 'tapajos' / 'eventos.csv.gz'
if lab.exists():
    ev = pd.read_csv(lab)
else:   # Etapa 2 ainda não rodou no Colab: só PRODES + DETER
    grid = rotulos.make_grid('tapajos')
    cache = config.LABELS_DIR / 'tapajos' / 'cache'
    ev = pd.concat([rotulos.overlay_events(grid, rotulos.fetch_prodes('tapajos', cache)),
                    rotulos.overlay_events(grid, rotulos.fetch_deter('tapajos', cache))], ignore_index=True)
ev = ev[ev['patch_id'].isin(pids)].copy()
ev['step'] = rotulos.step_of(ev['date_end'])
normal = rotulos.normal_flags(ev, np.array(pids)).set_index('patch_id')['normal_treino']

clear = dados.clear_fraction(root, 's2')
examples = {}
mid = [s for s in steps if '2020Q1' <= s <= '2024Q4']        # evento no meio da série: antes e depois visíveis
for tipo in ['desmatamento', 'mineracao', 'fogo']:
    e = ev[(ev['source'] == 'deter') & (ev['tipo'] == tipo) & ev['step'].isin(mid)]
    e = e.groupby(['patch_id', 'step'])['frac'].sum().sort_values(ascending=False)
    e = e[[clear[pids.index(p), steps.index(s)] >= config.MIN_CLEAR_FRAC for p, s in e.index]]   # evento visível
    if len(e):
        (pid, step), frac = e.index[0], e.iloc[0]
        examples[f'{tipo} ({frac:.0%} do patch, DETER {step})'] = (pids.index(pid), steps.index(step))
i_norm = int(np.argmax(np.where(normal.reindex(pids).values, clear.mean(axis=1), -1)))
examples['normal (sem evento 2019–2025)'] = (i_norm, None)
for k, (i, t) in examples.items():
    print(f'{k:<45} patch {pids[i]}  limpo médio {clear[i].mean():.0%}')

## 3. Séries temporais: RGB (S2) e VV (S1)

Cinza claro = trimestre sem observação limpa. Moldura escura = trimestre do alerta DETER.

In [ ]:
import matplotlib.pyplot as plt
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e1e0d9'
S2_COLOR, S1_COLOR = '#2a78d6', '#eb6834'      # slots categóricos 1 e 2, sempre nessa ordem
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': GRID, 'axes.labelcolor': MUTED, 'xtick.color': MUTED,
                     'ytick.color': MUTED, 'axes.spines.top': False, 'axes.spines.right': False})

def rgb(i, t):
    x = root['s2'][i, t, [3, 2, 1]].astype(np.float32) / 10000      # B4, B3, B2
    img = np.clip(np.moveaxis(x, 0, -1) * 3.5, 0, 1)
    img[root['s2_nobs'][i, t] == 0] = 0.88
    return img

def vv(i, t):
    x = root['s1'][i, t, 0].astype(np.float32)
    img = np.clip((x + 25) / 25, 0, 1)
    return np.where(np.isnan(x), 0.88, img)

T = len(steps)
fig, axes = plt.subplots(2 * len(examples), T, figsize=(T * 0.62, 2 * len(examples) * 0.78))
for r, (name, (i, te)) in enumerate(examples.items()):
    for t in range(T):
        for k, (fn, kw) in enumerate([(rgb, {}), (vv, {'cmap': 'gray', 'vmin': 0, 'vmax': 1})]):
            ax = axes[2 * r + k, t]
            ax.imshow(fn(i, t), **kw); ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_visible(t == te); sp.set_color(INK); sp.set_linewidth(2.5)
            if 2 * r + k == 2 * len(examples) - 1 and t % 4 == 0:
                ax.set_xlabel(steps[t], fontsize=7)
    axes[2 * r, 0].set_ylabel('S2', rotation=0, ha='right', va='center')
    axes[2 * r + 1, 0].set_ylabel('S1 VV', rotation=0, ha='right', va='center')
    axes[2 * r, T // 2].set_title(name, color=INK, fontsize=9, loc='center')
fig.subplots_adjust(left=0.04, right=0.995, top=0.95, bottom=0.05, wspace=0.05, hspace=0.45)
fig.savefig(run_dir / 'series_eventos.png', dpi=110); plt.show()

## 4. Normalização TerraMind

Estatísticas dos valores padronizados em pixels limpos de 40 patch-trimestres sorteados. As estatísticas de
pré-treino são globais; na Amazônia (floresta escura no visível, brilhante no NIR) espera-se média
negativa no visível e próxima de 0 no NIR, mas sem nada além de |média| ≈ 2.

In [ ]:
rng = np.random.default_rng(config.SEED)
ok = np.argwhere(clear >= config.MIN_CLEAR_FRAC)
pick = ok[rng.choice(len(ok), min(40, len(ok)), replace=False)]
rows = []
for mod, key, bands in [('S2L2A', 's2', config.S2_BANDS), ('S1GRD', 's1', config.S1_BANDS)]:
    vals = []
    for i, t in pick:
        x = dados.normalize(root[key][i, t], mod)
        m = root[f'{key}_nobs'][i, t] > 0
        vals.append(x[:, m])
    v = np.concatenate(vals, axis=1)
    for b, name in enumerate(bands):
        rows.append({'modalidade': mod, 'banda': name, 'media': np.nanmean(v[b]), 'desvio': np.nanstd(v[b]),
                     'p1': np.nanpercentile(v[b], 1), 'p99': np.nanpercentile(v[b], 99)})
norm = pd.DataFrame(rows)
display(norm.round(2))
bad = norm[norm['media'].abs() > 2]
print('OK: nenhuma banda com |média| > 2' if bad.empty else f'CONFERIR unidades destas bandas:\n{bad}')
norm.to_csv(run_dir / 'normalizacao.csv', index=False)

In [ ]:
(run_dir / 'resumo.json').write_text(json.dumps(
    {'exemplos': {k: {'patch': pids[i], 'passo_evento': steps[t] if t is not None else None} for k, (i, t) in examples.items()},
     'normalizacao_ok': bool(bad.empty)}, indent=2), encoding='utf-8')
print('Salvo em', run_dir)